# P8 Notebook 4 — Cross-dataset Theorem 1 Verification (Phase B)

**Goal**: Verify Theorem 1 on **VOC + INRIA** (50 train images each, matching phaseA sample), using PnPLO-trained detectors + denoisers. Reuse `C_PnPLO` ∈ [0.10, 0.26] (no refit) — strongest claim.

**Convention**: BGR end-to-end (matches PnPLO 120-cell pipeline). Re-measures ρₖ in BGR-luminance domain.

**Workflow** (run cells in order):
1. **Setup** — load 15 deep denoise models + define all denoising functions (BGR convention, verbatim from `YOLO_Denoise_Experiment_Karthy` Cells 9 + 13).
2. **Build denoised dataset** — generate noisy + denoised JPEGs for 50 train × 5 σ × 5 method × 2 dataset = 2500 ops/dataset. Idempotent.
3. **Re-measure ρₖ** phase-aware on the new JPEGs → `rho_per_cell__{ds}__xdataset_BGR__v1.csv`.
4. **Run val() + F1 from CM** with `classes=[0]` filter → `theorem_metrics_xdataset.csv`.

After Kiệt uploads the 2 output CSVs, Barry does offline analysis: join with α profiles (Table VI), compute S = Σ αₖ(1-ρₖ)², √(C·S), count violations, extend Fig 1 to 3-dataset overlay, draft new §V-G.

**Compute estimate**: ~1.5–2 hours on A100 (mostly Cell 2 denoising + Cell 4 val()).

---

In [ ]:
# ============================================================================
# Cell 1: Setup — load models, define denoise + noise generation
# ============================================================================
from pathlib import Path
import os, gc, hashlib, time, shutil, tempfile
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import bm3d
from torchvision import transforms
from scipy import fft as sp_fft

# Mount Drive if Colab
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_ROOT = Path('/content/drive/MyDrive/DOCTOR_PHD/FINAL PROJECT')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ---- Detectors (PnPLO-trained) ----
TRAINING_RUNS = DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/yolo_denoise_experiment/training_runs'
DETECTORS = ['yolov8m', 'yolov9m', 'yolov10m', 'yolo11m', 'yolo12m']
def clean_ckpt(det): return TRAINING_RUNS / f'{det}_noisy_noise_0' / 'weights' / 'best.pt'

# ---- Deep denoise models (15 = 3 archs × 5 σ) ----
DENOISE_MODELS_DIR = DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/yolo_denoise_experiment/denoise_models'

# ---- Datasets (50 train imgs each, from phaseA sample) ----
DATASETS_X = {
    'VOC': {
        'phaseA_csv': DRIVE_ROOT / '04_1_VOC_experiment/P8_outputs/rho_per_cell__VOC__phaseA__v1.csv',
        'clean_dir':  DRIVE_ROOT / '04_1_VOC_experiment/VOC_person/dataset_yolo/train/images',
        'label_dir':  DRIVE_ROOT / '04_1_VOC_experiment/VOC_person/dataset_yolo/train/labels',
        'out_root':   DRIVE_ROOT / '04_1_VOC_experiment/xdataset_BGR_v1',
        'classes_filter': [0], 'nc': 1, 'names': {0: 'person'},
    },
    'INRIA': {
        'phaseA_csv': DRIVE_ROOT / '04_2_INRIA_experiment/P8_outputs/rho_per_cell__INRIA__phaseA__v1.csv',
        'clean_dir':  DRIVE_ROOT / '04_2_INRIA_experiment/INRIA Person/train/images',
        'label_dir':  DRIVE_ROOT / '04_2_INRIA_experiment/INRIA Person/train/labels',
        'out_root':   DRIVE_ROOT / '04_2_INRIA_experiment/xdataset_BGR_v1',
        'classes_filter': [0], 'nc': 1, 'names': {0: 'person'},
    },
}

DENOISE_METHODS = ['bm3d', 'gaussian_filter', 'autoencoder', 'dncnn', 'cae_pso']
DEEP_METHODS    = ['autoencoder', 'dncnn', 'cae_pso']
SIGMAS = [1, 5, 10, 20, 30]

# ============ Deep architectures (verbatim from NB1 Cell 4) ============
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64,64,3,padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128,128,3,padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128,128,2,stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(128,64,3,padding=1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64,64,2,stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(64,3,3,padding=1), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

class DnCNN(nn.Module):
    def __init__(self, channels=3, num_layers=17, features=64):
        super().__init__()
        layers = [nn.Conv2d(channels, features, 3, padding=1, bias=False), nn.ReLU(inplace=True)]
        for _ in range(num_layers-2):
            layers += [nn.Conv2d(features,features,3,padding=1,bias=False),
                       nn.BatchNorm2d(features), nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(features,channels,3,padding=1,bias=False))
        self.layers = nn.Sequential(*layers)
    def forward(self, x): return x - self.layers(x)

class CAE(nn.Module):
    def __init__(self, base_filters=64, kernel_size=3):
        super().__init__()
        pad = kernel_size//2; f1, f2 = base_filters, base_filters*2
        self.encoder = nn.Sequential(
            nn.Conv2d(3,f1,kernel_size,padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.Conv2d(f1,f1,kernel_size,padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2),
            nn.Conv2d(f1,f2,kernel_size,padding=pad), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.Conv2d(f2,f2,kernel_size,padding=pad), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2))
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(f2,f2,2,stride=2), nn.BatchNorm2d(f2), nn.ReLU(inplace=True),
            nn.Conv2d(f2,f1,kernel_size,padding=pad), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(f1,f1,2,stride=2), nn.BatchNorm2d(f1), nn.ReLU(inplace=True),
            nn.Conv2d(f1,3,kernel_size,padding=pad), nn.Sigmoid())
    def forward(self, x): return self.decoder(self.encoder(x))

def _infer_cae(sd):
    w = sd.get('encoder.0.weight')
    return {'base_filters': w.shape[0], 'kernel_size': w.shape[2]} if w is not None else {'base_filters':64,'kernel_size':3}

def _build(method, sd):
    if method == 'autoencoder': m = DenoisingAutoencoder()
    elif method == 'dncnn':     m = DnCNN(channels=3, num_layers=17, features=64)
    elif method == 'cae_pso':   m = CAE(**_infer_cae(sd))
    else: raise ValueError(method)
    m.load_state_dict(sd)
    return m.eval().to(DEVICE)

# Load 15 deep models
denoise_models = {m: {} for m in DEEP_METHODS}
print(f'\nLoading 15 deep denoise models from {DENOISE_MODELS_DIR.name}/ ...')
for method in DEEP_METHODS:
    for s in SIGMAS:
        p = DENOISE_MODELS_DIR / f'{method}_sigma{s}.pt'
        if not p.exists():
            print(f'  ❌ MISSING: {p.name}'); continue
        try:    sd = torch.load(p, map_location=DEVICE, weights_only=True)
        except: sd = torch.load(p, map_location=DEVICE, weights_only=False)
        denoise_models[method][s] = _build(method, sd)
        print(f'  ✓ {method}_sigma{s}.pt')
total = sum(len(v) for v in denoise_models.values())
assert total == 15, f'Loaded {total}/15'
print(f'\n✓ {total}/15 deep models on {DEVICE}\n')

# ============ Denoise functions (BGR convention — big notebook Cells 9+13) ============
def denoise_gaussian_filter(img_bgr, sigma):
    """BGR uint8 in/out. Adaptive kernel by σ (big notebook Cell 9)."""
    if sigma is None or sigma <= 1: ksize = 3
    elif sigma <= 5:  ksize = 5
    elif sigma <= 15: ksize = 7
    else:             ksize = 9
    return cv2.GaussianBlur(img_bgr, (ksize, ksize), 0)

def denoise_bm3d(img_bgr, sigma=10):
    """BGR uint8 in/out. bm3d.bm3d_rgb on float [0,1] (big notebook Cell 9)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_float = img_rgb.astype(np.float64) / 255.0
    psd = max(sigma/255.0, 0.01)
    out = bm3d.bm3d_rgb(img_float, sigma_psd=psd)
    out = np.clip(out*255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)

def denoise_with_model_fast(image_bgr, model, patch_size=128, stride=112, batch_size=64):
    """Deep denoiser, BGR uint8 in/out, sliding window (big notebook Cell 13)."""
    model.eval()
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w, c = img_rgb.shape
    pad_h = (patch_size - h % patch_size) % patch_size
    pad_w = (patch_size - w % patch_size) % patch_size
    img_padded = np.pad(img_rgb, ((0,pad_h),(0,pad_w),(0,0)), mode='reflect')
    ph, pw = img_padded.shape[:2]
    result = np.zeros((ph,pw,c), dtype=np.float64)
    count  = np.zeros((ph,pw,1), dtype=np.float64)
    tx = transforms.ToTensor()
    patches, positions = [], []
    for y in range(0, ph-patch_size+1, stride):
        for x in range(0, pw-patch_size+1, stride):
            patches.append(tx(img_padded[y:y+patch_size, x:x+patch_size]))
            positions.append((y, x))
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for i in range(0, len(patches), batch_size):
            batch = torch.stack(patches[i:i+batch_size]).to(DEVICE)
            outs = model(batch).cpu().numpy()
            for j, (y, x) in enumerate(positions[i:i+batch_size]):
                result[y:y+patch_size, x:x+patch_size] += outs[j].transpose(1, 2, 0)
                count [y:y+patch_size, x:x+patch_size] += 1
    count = np.maximum(count, 1)
    result = (result / count)[:h,:w]
    result = np.clip(result*255, 0, 255).astype(np.uint8)
    return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

def apply_denoise(method, sigma, img_bgr):
    """Single dispatcher, BGR uint8 in/out."""
    if method == 'bm3d':            return denoise_bm3d(img_bgr, sigma)
    if method == 'gaussian_filter': return denoise_gaussian_filter(img_bgr, sigma)
    return denoise_with_model_fast(img_bgr, denoise_models[method][sigma])

# ============ Noise gen (deterministic, MD5 seed — matches NB1 Cell 6) ============
def add_noise(clean_bgr, sigma, fname):
    h = hashlib.md5(f'{fname}_{sigma}'.encode()).hexdigest()
    rng = np.random.RandomState(int(h[:8], 16))
    noise = rng.randn(*clean_bgr.shape) * sigma
    return np.clip(clean_bgr.astype(np.float64) + noise, 0, 255).astype(np.uint8)

# ============ Smoke test ============
print('Smoke test: 1 sample img × 5 methods at σ=20:')
sample_csv = pd.read_csv(DATASETS_X['VOC']['phaseA_csv'])
sample_name = sorted(sample_csv['image'].unique())[0]
sample_clean = cv2.imread(str(DATASETS_X['VOC']['clean_dir'] / sample_name))
assert sample_clean is not None, f'cannot read {sample_name}'
print(f'  Sample: {sample_name}, shape={sample_clean.shape}, dtype={sample_clean.dtype}')
noisy = add_noise(sample_clean, 20, sample_name)
for m in DENOISE_METHODS:
    t0 = time.time()
    out = apply_denoise(m, 20, noisy)
    assert out.shape == sample_clean.shape and out.dtype == np.uint8
    print(f'  ✓ {m:16}  {time.time()-t0:.2f}s  out_range=[{out.min()}, {out.max()}]')
print('\n✓ Setup complete')

In [ ]:
# ============================================================================
# Cell 2: Build denoised dataset (BGR convention)
#   For each (dataset, method, σ, image): noisy = add_noise(clean); denoised = apply_denoise(noisy)
#   Save: {out_root}/denoised/{method}_noise_{σ}/images/train/*.jpg
#         {out_root}/denoised/{method}_noise_{σ}/labels/train/*.txt (copied from cfg['label_dir'])
#   Plus baseline clean copy: {out_root}/clean/images/train/ + labels/train/
#   Idempotent — re-runs skip existing files.
# ============================================================================
for ds_name, cfg in DATASETS_X.items():
    print(f'\n=== Build denoised dataset: {ds_name} ===')
    phaseA = pd.read_csv(cfg['phaseA_csv'])
    img_names = sorted(phaseA['image'].unique())
    print(f'  {len(img_names)} images from phaseA sample')
    
    # 1. Copy clean baseline (for F1_clean later)
    clean_img_out = cfg['out_root'] / 'clean' / 'images' / 'train'
    clean_lbl_out = cfg['out_root'] / 'clean' / 'labels' / 'train'
    clean_img_out.mkdir(parents=True, exist_ok=True)
    clean_lbl_out.mkdir(parents=True, exist_ok=True)
    copied = 0
    for fname in img_names:
        src_i = cfg['clean_dir'] / fname; dst_i = clean_img_out / fname
        if src_i.exists() and not dst_i.exists():
            shutil.copy2(src_i, dst_i); copied += 1
        src_l = cfg['label_dir'] / (Path(fname).stem + '.txt')
        dst_l = clean_lbl_out / src_l.name
        if src_l.exists() and not dst_l.exists():
            shutil.copy2(src_l, dst_l)
    print(f'  ✓ clean baseline: {copied} new files copied (rest already present)')
    
    # 2. Denoise pipeline
    out_root = cfg['out_root'] / 'denoised'
    out_root.mkdir(parents=True, exist_ok=True)
    
    for method in DENOISE_METHODS:
        for sigma in SIGMAS:
            cell_dir = out_root / f'{method}_noise_{sigma}'
            img_out = cell_dir / 'images' / 'train'
            lbl_out = cell_dir / 'labels' / 'train'
            img_out.mkdir(parents=True, exist_ok=True)
            lbl_out.mkdir(parents=True, exist_ok=True)
            
            existing = {p.name for p in img_out.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'}}
            todo = [n for n in img_names if n not in existing]
            if not todo:
                print(f'  ⏭️  {method:16} σ={sigma:<3} 50/50 đã có')
                continue
            
            t0 = time.time()
            ok = 0
            for fname in todo:
                clean_path = cfg['clean_dir'] / fname
                if not clean_path.exists():
                    print(f'  ⚠ missing clean: {fname}'); continue
                clean_bgr = cv2.imread(str(clean_path))
                if clean_bgr is None: continue
                noisy    = add_noise(clean_bgr, sigma, fname)
                denoised = apply_denoise(method, sigma, noisy)
                cv2.imwrite(str(img_out / fname), denoised)
                # copy label
                lbl_src = cfg['label_dir'] / (Path(fname).stem + '.txt')
                lbl_dst = lbl_out / lbl_src.name
                if lbl_src.exists() and not lbl_dst.exists():
                    shutil.copy2(lbl_src, lbl_dst)
                ok += 1
            print(f'  ✓ {method:16} σ={sigma:<3}  {ok}/{len(todo)} imgs  ({time.time()-t0:.1f}s)')

print('\n✓ All denoised datasets built')

In [ ]:
# ============================================================================
# Cell 3: Compute ρₖ phase-aware on the new denoised JPEGs (BGR luminance domain)
#   For each (dataset, method, σ, image): read clean BGR + denoised BGR JPEG,
#                                          project to ITU-R BT.601 luminance,
#                                          compute rho_phase_k (Definition 4 in paper).
#   Output: rho_per_cell__{dataset}__xdataset_BGR__v1.csv
# ============================================================================

# BFPI library (verbatim from NB1 Cell 4, 1-channel paths inlined)
def get_band_boundaries(K=4):
    b = np.linspace(0, np.pi, K+1); b[-1] = np.inf; return b

def radial_frequency_map(H, W):
    fy = np.fft.fftshift(np.fft.fftfreq(H)) * 2*np.pi
    fx = np.fft.fftshift(np.fft.fftfreq(W)) * 2*np.pi
    fxx, fyy = np.meshgrid(fx, fy, indexing='xy')
    return np.sqrt(fxx**2 + fyy**2)

def _band_E(img1ch, boundaries, rm):
    F = np.fft.fftshift(sp_fft.fft2(img1ch.astype(np.float64)))
    power = np.abs(F)**2
    return np.array([power[(rm >= boundaries[k]) & (rm < boundaries[k+1])].sum()
                     for k in range(len(boundaries)-1)])

def _band_D(clean1ch, den1ch, boundaries, rm):
    Fc = np.fft.fftshift(sp_fft.fft2(clean1ch.astype(np.float64)))
    Fd = np.fft.fftshift(sp_fft.fft2(den1ch.astype(np.float64)))
    dpow = np.abs(Fc - Fd)**2
    return np.array([dpow[(rm >= boundaries[k]) & (rm < boundaries[k+1])].sum()
                     for k in range(len(boundaries)-1)])

def compute_rho_phase_1ch(clean, denoised, boundaries, rm, eps=1e-10):
    E = _band_E(clean, boundaries, rm)
    D = _band_D(clean, denoised, boundaries, rm)
    return 1.0 - np.sqrt(D / (E + eps))

def bgr_to_luminance(bgr_uint8):
    """ITU-R BT.601 from cv2-BGR uint8: Y = 0.299·R + 0.587·G + 0.114·B."""
    b = bgr_uint8.astype(np.float64)
    return 0.299*b[..., 2] + 0.587*b[..., 1] + 0.114*b[..., 0]

BOUNDARIES = get_band_boundaries(K=4)
print(f'BFPI K=4, boundaries: {BOUNDARIES}\n')

# Cache radial_map by (H, W) — images may have varying sizes
_rm_cache = {}
def _get_rm(H, W):
    key = (H, W)
    if key not in _rm_cache:
        _rm_cache[key] = radial_frequency_map(H, W)
    return _rm_cache[key]

for ds_name, cfg in DATASETS_X.items():
    print(f'=== ρₖ phase-aware: {ds_name} ===')
    out_csv = cfg['out_root'] / f'rho_per_cell__{ds_name}__xdataset_BGR__v1.csv'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    
    # Resume-safe
    rows = []
    done = set()
    if out_csv.exists():
        prev = pd.read_csv(out_csv)
        rows = prev.to_dict('records')
        done = {(r['method'], int(r['sigma']), r['image']) for r in rows}
        print(f'  resume: {len(done)} rows done')
    
    phaseA = pd.read_csv(cfg['phaseA_csv'])
    img_names = sorted(phaseA['image'].unique())
    
    t0 = time.time()
    n_new = 0
    for method in DENOISE_METHODS:
        for sigma in SIGMAS:
            cell_dir = cfg['out_root'] / 'denoised' / f'{method}_noise_{sigma}' / 'images' / 'train'
            todo = [n for n in img_names if (method, sigma, n) not in done]
            if not todo: continue
            for fname in todo:
                clean_path = cfg['clean_dir'] / fname
                den_path   = cell_dir / fname
                if not (clean_path.exists() and den_path.exists()):
                    continue
                clean_bgr = cv2.imread(str(clean_path))
                den_bgr   = cv2.imread(str(den_path))
                if clean_bgr is None or den_bgr is None: continue
                if clean_bgr.shape != den_bgr.shape:
                    H, W = clean_bgr.shape[:2]
                    den_bgr = cv2.resize(den_bgr, (W, H))
                clean_lum = bgr_to_luminance(clean_bgr)
                den_lum   = bgr_to_luminance(den_bgr)
                H, W = clean_lum.shape
                rho = compute_rho_phase_1ch(clean_lum, den_lum, BOUNDARIES, _get_rm(H, W))
                rows.append({
                    'dataset': ds_name, 'sigma': sigma, 'method': method, 'image': fname,
                    'rho_phase_1': rho[0], 'rho_phase_2': rho[1],
                    'rho_phase_3': rho[2], 'rho_phase_4': rho[3],
                })
                n_new += 1
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            print(f'  ✓ {method:16} σ={sigma:<3} ({(time.time()-t0)/60:.1f}m)')
    print(f'  → {n_new} new rows, total {len(rows)} → {out_csv}\n')

print('✓ ρₖ phase-aware computation complete')

In [ ]:
# ============================================================================
# Cell 4: Run val() + F1 from CM (classes=[0] for VOC/INRIA)
#   Output: theorem_metrics_xdataset.csv  (extends theorem_metrics schema with 'dataset' column)
#   F1-from-CM logic VERBATIM NB3 Cell 15: handles nc<2 (VOC/INRIA single-class).
# ============================================================================
from ultralytics import YOLO
import yaml

OUT_CSV_X = DRIVE_ROOT / '04_RESULT_TRAIN_KARTHY/YOLO_Denoise_Experiment_Karthy/P8_outputs/theorem_metrics_xdataset.csv'
OUT_CSV_X.parent.mkdir(parents=True, exist_ok=True)
EVAL_CONF = 0.25
EVAL_IOU  = 0.50
TMP_RUNS = '/tmp/xds_runs'

def _write_yaml(root_dir, ds_cfg):
    """root_dir contains images/train/ + labels/train/."""
    yp = Path(root_dir) / 'data.yaml'
    yaml.dump({'path': str(Path(root_dir).resolve()),
               'train': 'images/train', 'val': 'images/train', 'test': 'images/train',
               'names': ds_cfg['names'], 'nc': ds_cfg['nc']},
              open(yp, 'w'))
    return yp

def eval_one(model, root_dir, ds_cfg, run_name):
    yp = _write_yaml(root_dir, ds_cfg)
    v = model.val(data=str(yp), split='test',
                  conf=EVAL_CONF, iou=EVAL_IOU,
                  plots=False, verbose=False, device=0,
                  classes=ds_cfg.get('classes_filter'),
                  project=TMP_RUNS, name=run_name, exist_ok=True)
    cm = v.confusion_matrix.matrix
    nc = cm.shape[0] - 1
    # VERBATIM NB3 Cell 15 logic
    if nc < 2:
        TP = int(round(cm[0,0])); FN = int(round(cm[0,1]))
        FP = int(round(cm[1,0])); TN = 0
    else:
        TP = int(round(cm[0,0])); FN = int(round(cm[0,1:].sum()))
        FP = int(round(cm[1,0])) + int(round(cm[2,0]))
        TN = int(round(cm[1,1])) + int(round(cm[1,2]))
    tot = TP + FP + FN + TN
    Sens = TP/(TP+FN) if (TP+FN) else 0
    Prec = TP/(TP+FP) if (TP+FP) else 0
    return {
        'Accuracy':   round((TP+TN)/tot, 4) if tot else 0,
        'F1-Score':   round(2*Prec*Sens/(Prec+Sens), 4) if (Prec+Sens) else 0,
        'Sensitivity':round(Sens, 4),
        'Specificity':round(TN/(TN+FP), 4) if (TN+FP) else 0,
        'Precision':  round(Prec, 4),
        'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
    }

# Resume-safe
rows = []
done = set()
if OUT_CSV_X.exists():
    prev = pd.read_csv(OUT_CSV_X)
    rows = prev.to_dict('records')
    done = {(r['dataset'], r['model'], r['denoise_method'], int(r['noise_sigma'])) for r in rows}
    print(f'Resume: {len(done)} rows done\n')

t0 = time.time()
for ds_name, cfg in DATASETS_X.items():
    for det in DETECTORS:
        ck = clean_ckpt(det)
        if not ck.exists():
            print(f'  ❌ missing checkpoint: {det}'); continue
        model = YOLO(str(ck))
        
        # baseline (clean)
        if (ds_name, det, 'clean', 0) not in done:
            try:
                m = eval_one(model, cfg['out_root'] / 'clean', cfg, f'{ds_name}_{det}_clean')
                rows.append(dict(dataset=ds_name, model=det, noise_sigma=0,
                                 denoise_method='clean', **m))
                pd.DataFrame(rows).to_csv(OUT_CSV_X, index=False)
                print(f'  [{ds_name}] {det:9} clean              F1={m["F1-Score"]:.4f}'
                      f'  ({(time.time()-t0)/60:.1f}m)')
            except Exception as e:
                print(f'  ERR {ds_name} {det} clean: {e}')
        
        # (denoiser, σ) cells
        for method in DENOISE_METHODS:
            for sigma in SIGMAS:
                if (ds_name, det, method, sigma) in done: continue
                cell_root = cfg['out_root'] / 'denoised' / f'{method}_noise_{sigma}'
                if not (cell_root / 'images' / 'train').exists():
                    print(f'  ⏭️  missing: {cell_root.name}'); continue
                try:
                    m = eval_one(model, cell_root, cfg, f'{ds_name}_{det}_{method}_s{sigma}')
                    rows.append(dict(dataset=ds_name, model=det, noise_sigma=sigma,
                                     denoise_method=method, **m))
                    pd.DataFrame(rows).to_csv(OUT_CSV_X, index=False)
                    print(f'  [{ds_name}] {det:9} {method:16} σ={sigma:<3} '
                          f'F1={m["F1-Score"]:.4f}  ({(time.time()-t0)/60:.1f}m)')
                except Exception as e:
                    print(f'  ERR {ds_name} {det} {method} σ{sigma}: {e}')
        
        del model; gc.collect(); torch.cuda.empty_cache()

print(f'\n✓ Done. {len(rows)} rows → {OUT_CSV_X}')